## ***PROJET DE MODELISATION 2 : MISE EN PLACE D'UNE APPLICATION DE SIMULATION***

Mise en place d'une interface de simulation d'un solide sollicité en Traction, Compression ou Cisallement (En python en utilisant Tkinter). On simulera dans ce TP soit un pavé, soit un solide cylindre dans le domaine élastique.

#### ***-Tkinter pour la visualisation***
#### ***-Python pour le backend***

***Paramètres d'entrées du matériau : E module d'Young, V module de poisson***

***Paramètres de géométrie : cylindre (rayon), pavé (longueur, largeur, hauteur)***

***Sollicitation souhaité : traction, compression, cisaillement***

#### A- IMPORT DES MODULES NÉCESSAIRES

In [1]:
# ------------------------------------------------------------
# Tkinter : interface graphique
# Numpy : calcul numérique
# Matplotlib : visualisation 3D
# PIL : gestion des images
# ------------------------------------------------------------

from time import time
import tkinter as tk
from tkinter import messagebox, simpledialog, ttk, constants
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, PolyCollection
from PIL import Image, ImageTk
from datetime import datetime

#### B- BASE DE DONNÉES DES MATÉRIAUX

In [2]:
# ------------------------------------------------------------
# Dictionnaire contenant :
# [masse volumique ρ, module d’Young E, coefficient de Poisson ν]
# ------------------------------------------------------------

liste_materiaux = {"Acier": [7850, 210, 0.30],
                  "Diamant": [3517, 1220, 0.2],
                  "Fer": [7860, 208, 0.29],
                  "Fontes": [7300, 170, 0.26],
                  "Laiton(Cu+Zn)": [8470, 130, 0.37],
                  "Aluminium": [2700, 69, 0.346],
                  "Or": [18900, 78, 0.42],
                  "Magnésium": [1740, 44, 0.35],
                  "Cuivre": [8920, 124, 0.33],
                  "Titane": [4500, 114, 0.34],
                  "Béton": [2500, 50, 0.20],
                  "Verre": [2320, 69, 0.3],
                  "Caoutchouc": [100, 76, 0.50],
                  "Polystyrène": [1050, 3, 0.4],
                  "Plomb": [11300, 15, 0.44],
                  "Zinc": [7140, 90, 0.33],
                  "Silicium": [2330, 130, 0.28]
                 }

# Tri alphabétique pour affichage propre
liste_materiaux = dict(sorted(liste_materiaux.items()))

### C- GESTION DE LA SIMULATION ET DE L'INTERFACE

#### CLASSE PRINCIPALE DE SIMULATION

In [3]:
class simulation:
 
    """
    Classe principale gérant :
    - L’interface graphique Tkinter
    - La définition du matériau
    - La géométrie du solide
    - Le maillage
    - L’application de la déformation
    - La visualisation 3D
    """ 
    
    def __init__(self, im=False):
        
        # ----------------------------------------------------
        # INITIALISATION DE LA FENÊTRE PRINCIPALE
        # ----------------------------------------------------
        self.interface = tk.Tk()
        self.interface.title("MODELISATION 2 : Interface de Simulation d'un solide")
        self.interface.geometry("3000x700")
        self.interface.config(bg="grey")
        self.im = im
    
         # ----------------------------------------------------
        # FRAME D’ACCUEIL
        # ----------------------------------------------------
        self.frame_principal = tk.Frame(self.interface, bg="white", relief="solid", borderwidth=15)
        self.frame_principal.place(relwidth=1, relheight=1)

        # --- Message d'accueil 
        tk.Button(self.frame_principal,
                  text=("Bienvenue sur notre interface de simulation d'un solide soumis à une solliciation\n"
                        "de traction, compression ou cisaillement dans le domaine élastique"),
                  font=("Tahoma", 20, "italic"),
                  borderwidth=10,
                  relief = "solid",
                  activebackground="gray",
                  activeforeground="white"
                 ).pack(pady=50)

        # ----------------------------------------------------
        # OPTION IMAGE DE FOND
        # ----------------------------------------------------
        if im:
            # ---- Image de fond ----
            img = Image.open("IM.jpeg")
            img = img.resize((1500, 800)) #Redimensionner pour la fenêtre
            photo = ImageTk.PhotoImage(img)
    
            bg_image = tk.Label(self.frame_principal, image=photo)
            bg_image.image = photo
            bg_image.place(x=1, y=1, relwidth=1, relheight=1)
    
        # ----------------------------------------------------
        # INITIALISATION DES VARIABLES DE SIMULATION
        # ----------------------------------------------------
        # --- Géométrie
        self.geo_choosing = None
        self.longueur = None
        self.largeur = None
        self.hauteur = None
        self.rayon = None
    
        # --- Matériau
        self.materiau = None
        self.module_dYoung = None
        self.coeff_poisson = None
        self.masse_volumique = None
    
        # --- Sollicitation
        self.sollicitation = None
        self.charge = None
    
        # --- Maillage
        self.noeuds = None
        self.elements = None
    
        # =============================
        # --- Visualisation graphique du solide
        self.fig = None
        self.ax = None
        self.canvas = None
        self.opac = 0.2
    
        # =============================
        # --- Frames secondaires
        self.frame_secondaire = None
        self.bar = None
        self.canvas_frame = None

    # ===========================================
    # GESTIONS DES BOUTONS DE L'INTERFACE
    # ===========================================
    def Addbouton(self):
         
        """
        Ajoute tous les boutons de commande latéraux :
        - Géométrie
        - Matériau
        - Sollicitation
        - Charge
        - Conditions aux limites
        - Maillage
        - Simulation
        - Nettoyage
        """
        
        bouton_list = [
            ("Géométrie", self.Geometrie),
            ("Matériau", self.Materiau),
            ("Sollicitation", self.Sollicitation),
            ("Charge (Newton)", self.Charge),
            ("Conditions\naux limites", getattr(self, "ConditionsLimites", None)),
            ("Maillage", self.Mesh),
            ("Simuler", self.appliquer_deformation),
            ("Nettoyer Tout", self.Clear)
        ]
        
        #-- Création des boutons sur l'interface 
        for txt, commd in bouton_list:
            ttk.Button(
                self.bar,
                text=txt,
                command=commd if commd is not None else lambda t=txt: messagebox.showinfo("Info", f"{t} non implémenté"),
                width=20,
                cursor="hand2"
            ).pack(pady=10)

        # Menu options avancées
        menu_btn = ttk.Menubutton(self.bar, width=16, text="Options Avancées", cursor="hand2")
        menu = tk.Menu(menu_btn, tearoff=0)
        menu.add_command(label="Charger fichier".upper(), font=("Times", 10, "bold"))
        menu.add_command(label="Sauvegarder fichier".upper(), font=("Times", 10, "bold"))
        menu.add_command(label="Retour".upper(), command=lambda: self.frame_principal.tkraise(), font=("Times", 10, "bold"))
        menu_btn.config(menu=menu)
        menu_btn.pack(pady=10, side="bottom")

        # ===========================================
        # CHOIX DU THEME D'AFFICHAGE
        # ===========================================
    def Mode(self):
        
        """
        Fenêtre modale permettant de choisir
        le thème graphique de l’application.
        """
        
        mode_frame = tk.Toplevel(self.interface)
        mode_frame.geometry("350x150")
        mode_frame.resizable(False, False)
        mode_frame.title("Choix du mode")
        mode_frame.transient(self.interface)
        mode_frame.grab_set()
        mode_frame.focus_force()
    
        tk.Label(mode_frame,
                 text="Choisissez le thème d'affichage",
                 font=("Tahoma", 11, "italic")
                ).pack(pady=10)
        
        mode = tk.StringVar(value="black")
    
        ttk.Combobox(
            mode_frame,
            textvariable=mode,
            values=["grey", "black", "blue", "skyblue"],
            state="readonly"
        ).pack(pady=10)
    
        def valider():
            # =============================
            # --- Nettoyage ancien affichage
            if self.frame_secondaire is not None:
                self.frame_secondaire.destroy()
    
            # =============================
            # --- Création frame secondaire
            self.frame_secondaire = tk.Frame(
                self.interface,
                bg=mode.get(),
                borderwidth=6,
                relief="groove"
            )
            self.frame_secondaire.pack(fill="both", expand=True)

            # Entête
            header = tk.Frame(self.frame_secondaire, bg=mode.get(), height=90, borderwidth=6, relief="groove")
            header.pack(fill=tk.X)
            tk.Label(header, text="📦 SOFTWARE SIMULATION 🛠️", font=("Arial", 30, "italic"), bg=mode.get(), fg="white").pack(pady=10) #1e3a8a
            # =============================
            # --- Barre latérale gauche
            self.bar = tk.Frame(
                self.frame_secondaire,
                width=260,
                bg=mode.get(),
                borderwidth=6,
                relief="groove"
            )
            self.bar.pack(side="left", fill="y")
            
    
            # =============================
            # --- Zone canvas
            self.canvas_frame = tk.Frame(
                self.frame_secondaire,
                bg=mode.get()
            )
            self.canvas_frame.bind("<Configure>", lambda event: self.canvas.draw())

            self.canvas_frame.pack(side="right", fill="both", expand=True)
            
            # =============================
            # --- Figure Matplotlib
            self.fig = plt.Figure(figsize=(9, 8), dpi=120)
            self.ax = self.fig.add_subplot(111, projection="3d")
            self.ax.set_axis_off()
            self.ax.margins(0)
    
            self.canvas = FigureCanvasTkAgg(self.fig, master=self.canvas_frame)
            self.canvas.draw()
            self.canvas.get_tk_widget().pack(fill="both", expand=True)
    
            # =============================
            # --- Boutons
            self.Addbouton()
    
            # =============================
            # --- Affichage et Nettoyage 
            self.frame_secondaire.tkraise()
            mode_frame.destroy()
    
        ttk.Button(mode_frame, text="Valider", command=valider, width=12, cursor="hand2").pack(pady=10)

    def Geometrie(self):
        if self.frame_secondaire is None:
            messagebox.showwarning("Attention", "Veuillez démarrer la simulation.")
            return
        #-- fenetre pour la géométrie
        geo_win = tk.Toplevel(self.frame_secondaire)
        geo_win.title("Choix de la géométrie")
        geo_win.geometry("350x190")
        geo_win.resizable(False, False)
        geo_win.grab_set()
    
        tk.Label(geo_win,
                 text="CHOIX DE LA GÉOMÉTRIE",
                 font=("Verdana", 12, "italic")
                ).pack(pady=10)
        #-- Par defaut
        geometrie = tk.StringVar(value="Pavé")
    
        ttk.Combobox(
            geo_win,
            textvariable=geometrie,
            values=["Pavé", "Cylindre"],
            state="readonly"
        ).pack(pady=10)
    
        def valider_choix():
            choix = geometrie.get()
            self.geo_choosing = choix
    
            # Réinitialisation complète des dimensions
            self.longueur = None
            self.largeur = None
            self.hauteur = None
            self.rayon = None
    
            geo_win.destroy()
    
            if choix == "Pavé":
                self.saisie_pave()
            elif choix == "Cylindre":
                self.saisie_cylindre()
            else:
                messagebox.showerror("Erreur", "Géométrie non reconnue")
    
        ttk.Button(
            geo_win,
            text="Valider",
            command=valider_choix,
            width=12,
            cursor="hand2"
        ).pack(pady=10)


    def saisie_pave(self):
        win = tk.Toplevel(self.frame_secondaire)
        win.title("Dimensions du pavé")
        win.geometry("450x250")
        win.resizable(False, False)
        win.grab_set()
    
        champs = {
            "Longueur (mm)": tk.Entry(win),
            "Largeur (mm)": tk.Entry(win),
            "Hauteur (mm)": tk.Entry(win),
        }
    
        for label, entry in champs.items():
            tk.Label(win, text=label, font=("Tahoma", 12, "italic")).pack(pady=5)
            entry.pack()
    
        def valider():
            try:
                self.longueur = float(champs["Longueur (mm)"].get())
                self.largeur = float(champs["Largeur (mm)"].get())
                self.hauteur = float(champs["Hauteur (mm)"].get())
    
                if min(self.longueur, self.largeur, self.hauteur) <= 0:
                    raise ValueError
                else:
                    win.destroy()
                    messagebox.showinfo("Succès", "Dimensions du pavé enregistrées")
        
                    self.Pavé()
    
            except ValueError:
                messagebox.showerror("Erreur", "Valeurs numériques strictement positives requises")
    
        ttk.Button(win, text="Valider", command=valider, width=15).pack(pady=10)


    def saisie_cylindre(self):
        win = tk.Toplevel(self.frame_secondaire)
        win.title("Dimensions du cylindre")
        win.geometry("350x220")
        win.resizable(False, False)
        win.grab_set()
    
        tk.Label(win, text="Rayon (mm)", font=("Tahoma", 12, "italic")).pack(pady=5)
        entry_r = tk.Entry(win)
        entry_r.pack()
    
        tk.Label(win, text="Hauteur (mm)", font=("Tahoma", 12, "italic")).pack(pady=5)
        entry_h = tk.Entry(win)
        entry_h.pack()
    
        def valider():
            try:
                self.rayon = float(entry_r.get())
                self.hauteur = float(entry_h.get())
    
                if self.rayon <= 0 or self.hauteur <= 0:
                    raise ValueError
                else:
                    win.destroy()
                    messagebox.showinfo("Succès", "Dimensions du cylindre enregistrées")
    
                    self.Cylindre()
    
            except ValueError:
                messagebox.showerror("Erreur", "Valeurs numériques strictement positives requises")
    
        ttk.Button(win, text="Valider", command=valider, width=15).pack(pady=10)


    def Materiau(self):
        if self.frame_secondaire is None:
            messagebox.showwarning("Attention", "Veuillez démarrer la simulation.")
            return
    
        if self.geo_choosing is None:
            messagebox.showwarning("Attention", "Veuillez définir une géométrie avant le matériau.")
            return
    
        win = tk.Toplevel(self.frame_secondaire)
        win.title("Choix du matériau")
        win.geometry("350x220")
        win.resizable(False, False)
        win.grab_set()
    
        tk.Label(
            win,
            text="CHOIX DU MATÉRIAU",
            font=("Tahoma", 11, "italic")
        ).pack(pady=10)
    
        materiaux = list(liste_materiaux.keys()) + ["Autre"]
        choix_mat = tk.StringVar(value=materiaux[0])
    
        ttk.Combobox(
            win,
            textvariable=choix_mat,
            values=materiaux,
            state="readonly"
        ).pack(pady=10)
    
        def valider():
            mat = choix_mat.get()
    
            try:
                if mat != "Autre":
                    rho, E, nu = liste_materiaux[mat]
    
                    self.masse_volumique = float(rho)
                    self.module_dYoung = float(E)
                    self.coeff_poisson = float(nu)
    
                else:
                    E = simpledialog.askfloat(
                        "Module d'Young",
                        "Entrez E (en GPa)"
                    )
                    nu = simpledialog.askfloat(
                        "Coefficient de Poisson",
                        "Entrez ν (0 < ν < 0.5)"
                    )
                    rho = simpledialog.askfloat(
                        "Masse volumique",
                        "Entrez ρ (kg/m³)"
                    )
    
                    if E is None or nu is None or rho is None:
                        raise ValueError
    
                    if E <= 0 or rho <= 0 or not (0 < nu < 0.5):
                        raise ValueError
                    else:
                        #Conversion GPa → Pa
                        self.module_dYoung = E
                        self.coeff_poisson = nu
                        self.masse_volumique = rho
    
                self.materiau = mat
                win.destroy()
                messagebox.showinfo("Succès", "Caractéristiques du matériau enregistrées")
    
                self._rafraichir_affichage()
    
            except ValueError:
                messagebox.showerror("Erreur", "Valeurs physiques invalides")
    
        ttk.Button(
            win,
            text="Valider",
            command=valider,
            width=15,
            cursor="hand2"
        ).pack(pady=10)


    def _rafraichir_affichage(self):
        if self.geo_choosing == "Pavé" and self.longueur is not None:
            self.Pavé(alpha=1, color='green')
        elif self.geo_choosing == "Cylindre" and self.rayon is not None:
            self.Cylindre(alpha=1, color='green')

    def Sollicitation(self):
        if self.frame_secondaire is None:
            messagebox.showwarning("Attention", "Veuillez démarrer la simulation.")
            return
    
        if self.geo_choosing is None:
            messagebox.showwarning("Attention", "Veuillez définir une géométrie avant la sollicitation.")
            return
    
        win = tk.Toplevel(self.frame_secondaire)
        win.title("Choix de la sollicitation")
        win.geometry("350x200")
        win.resizable(False, False)
        win.grab_set()
    
        tk.Label(
            win,
            text="CHOIX DE LA SOLLICITATION",
            font=("Verdana", 12, "italic")
        ).pack(pady=10)
    
        sollicitation = tk.StringVar(value="Traction")
    
        ttk.Combobox(
            win,
            textvariable=sollicitation,
            values=["Traction", "Compression", "Cisaillement"],
            state="readonly"
        ).pack(pady=10)
    
        def valider():
            self.sollicitation = sollicitation.get()
            win.destroy()
            messagebox.showinfo("Succès", f"Sollicitation définie : {self.sollicitation}")
    
        ttk.Button(
            win,
            text="Valider",
            command=valider,
            width=15,
            cursor="hand2"
        ).pack(pady=10)


    def Charge(self):
        if self.frame_secondaire is None:
            messagebox.showwarning("Attention", "Veuillez démarrer la simulation.")
            return
    
        if self.sollicitation is None:
            messagebox.showwarning("Attention", "Veuillez définir la sollicitation avant la charge.")
            return
    
        if self.geo_choosing is None:
            messagebox.showwarning("Attention", "Veuillez définir une géométrie avant la charge.")
            return
    
        force = simpledialog.askfloat(
            "Charge appliquée",
            "Valeur de la force appliquée (en Newton N)",
            minvalue=0.0
        )
    
        if force is None:
            return  # On annule tout
    
        if force <= 0:
            messagebox.showerror("Erreur", "La charge doit être strictement positive.")
            return
        else:
            if self.charge!=None:
                self.charge = force
                messagebox.showinfo("Succès",f"Charge modifiée : {self.charge} N")
            else:
                self.charge = force
                messagebox.showinfo("Succès",f"Charge enregistrée : {self.charge} N")


    def ConditionsLimites(self):
        if self.frame_secondaire is None:
            messagebox.showwarning("Attention", "Veuillez démarrer la simulation.")
            return
    
        if self.geo_choosing is None:
            messagebox.showwarning("Attention", "Veuillez définir une géométrie avant les conditions aux limites.")
            return
    
        win = tk.Toplevel(self.frame_secondaire)
        win.title("Conditions aux limites")
        win.geometry("360x260")
        win.resizable(False, False)
        win.grab_set()
    
        tk.Label(
            win,
            text="CONDITIONS AUX LIMITES",
            font=("Verdana", 12, "italic")
        ).pack(pady=10)
    
        # Type de condition (pour l'instant unique)
        type_cl = "Encastrement"
    
        tk.Label(win, text="Type : Encastrement").pack(pady=5)
        
        faces = ["x (bleu)", "y (rouge)", "z (vert)"]
        face_choisie = tk.StringVar(value=faces[0])
    
        ttk.Combobox(
            win,
            textvariable=face_choisie,
            values=faces,
            state="readonly"
        ).pack(pady=10)

        def valider():
            self.conditions_limites = {
                "type": type_cl,
                "face": faces.index(face_choisie.get()),
                "ddl": ["Ux", "Uy", "Uz"]
            }
    
            win.destroy()
            messagebox.showinfo(
                "Succès",
                f"Condition aux limites définie sur la face {face_choisie.get()}"
            )
    
        ttk.Button(
            win,
            text="Valider",
            command=valider,
            width=15,
            cursor="hand2"
        ).pack(pady=15)

    # ===========================================
    # DESSIN DES GEOMETRIES
    # ===========================================

    def Pavé(self, alpha=None, color = None):

        """
        Génère et affiche un pavé 3D.
        - Calcule l’aire de section
        - Définit les sommets
        - Affiche les faces 3D
        """
        
        # Opacité personnalisée
        if alpha != None:
            self.opac = alpha
        col = 'grey'
        if color != None:
            col = color
        # Vérification dimensions
        longueur = getattr(self, 'longueur', 0)
        largeur = getattr(self, 'largeur', 0)
        hauteur = getattr(self, 'hauteur', 0)
    
        if longueur <= 0 or largeur <= 0 or hauteur <= 0:
            messagebox.showerror("Problème", "Dimensions invalides")
            return
        self.aire = longueur * largeur
        # Nettoyer l'axe avant de dessiner
        self.ax.cla()
        self.ax.margins(0)
        self.ax.set_axis_off()
    
        # Définir les sommets du pavé
        sommets = np.array([
            [0, 0, 0], [longueur, 0, 0], [longueur, largeur, 0], [0, largeur, 0],
            [0, 0, hauteur], [longueur, 0, hauteur], [longueur, largeur, hauteur], [0, largeur, hauteur]
        ])
    
        # Faces du pavé
        faces = [
            [sommets[0], sommets[1], sommets[5], sommets[4]],
            [sommets[1], sommets[2], sommets[6], sommets[5]],
            [sommets[2], sommets[3], sommets[7], sommets[6]],
            [sommets[3], sommets[0], sommets[4], sommets[7]],
            [sommets[0], sommets[1], sommets[2], sommets[3]],
            [sommets[4], sommets[5], sommets[6], sommets[7]]
        ]
    
        # Création de la collection 3D
        pave = Poly3DCollection(faces, facecolor=col, edgecolor='black', linewidth=1, alpha=self.opac)
        self.ax.add_collection3d(pave)
    
        """# Petit repère dans le coin
        repere_pts = np.array([[0,0,0],[longueur*0.2,0,0],[0,largeur*0.2,0],[0,0,hauteur*0.2]])
        lignes = [[repere_pts[0], repere_pts[1]], [repere_pts[0], repere_pts[2]], [repere_pts[0], repere_pts[3]]]
        self.ax.add_collection3d(Poly3DCollection(lignes, edgecolor='blue', linewidth=1, alpha=0.9))"""
    
        # Ajuster les limites et proportions
        """max_dim = max(longueur, largeur, hauteur)
        self.ax.set_box_aspect([longueur/max_dim, largeur/max_dim, hauteur/max_dim])
        self.ax.set_xlim(0, longueur)
        self.ax.set_ylim(0, largeur)
        self.ax.set_zlim(0, hauteur)"""

        self.ax.set_box_aspect([longueur, largeur, hauteur])  # ou pour le cylindre: [1,1,H/R]
        self.ax.set_xlim(-longueur*0.1, longueur*1.1)
        self.ax.set_ylim(-largeur*0.1, largeur*1.1)
        self.ax.set_zlim(0, hauteur*1.1)

        # Label 
        self.ax.set_xlabel('X (mm)')
        self.ax.set_ylabel('X (mm)')
        self.ax.set_zlabel('X (mm)')

        # Si on est dans Tkinter, utilise canvas.draw() au lieu de plt.show()
        if hasattr(self, 'canvas'):
            self.canvas.draw()
            self.afficher_reperes(taille=0.3)

        else:
            plt.show()

    def afficher_reperes(self, taille=0.1):
        # Crée le repère relatif au solide
        x, y, z = 0, 0, 0  # origine du repère
        point = np.array([
            [x, y, z],
            [x + taille, y, z],  # X
            [x, y + taille, z],  # Y
            [x, y, z + taille]   # Z
        ])
        points = np.array([[-2, 0, 0], [-1, 0, 0], [-2, 1, 0], [-2, 0, 1]])
        lignes = [
            [points[0], points[1]],
            [points[0], points[2]],
            [points[0], points[3]]
        ]
        
        # Crée les Poly3DCollection pour chaque axe
        self.reperes = []
        color = ("blue", "red", "green")
        for i, ligne in enumerate(lignes):
            rep = Poly3DCollection([ligne], edgecolor=color[i], linewidth=1.2, alpha=1)
            self.reperes.append(rep)
            self.ax.add_collection3d(rep)



    def Cylindre(self, alpha=None, color=None):

        """
        Génère et affiche un cylindre 3D.
        - Calcule l’aire de section
        - Génère surfaces latérales et bases
        """
        
        # Opacité personnalisée
        if alpha is not None:
            self.opac = alpha
        col = 'grey'
        if color != None:
            col = color
        rayon = getattr(self, 'rayon', 0)
        hauteur = getattr(self, 'hauteur', 0)
    
        if rayon <= 0 or hauteur <= 0:
            messagebox.showerror("Problème", "Dimensions invalides")
            return
        self.aire = np.pi * rayon * rayon
        # Nettoyage de l'axe
        self.ax.cla()
        self.ax.set_axis_off()
    
        # Surface latérale
        n_theta = 20
        n_z = 10
        theta = np.linspace(0, 2*np.pi, n_theta)
        z = np.linspace(0, hauteur, n_z)
        theta_grid, z_grid = np.meshgrid(theta, z)
        x = rayon * np.cos(theta_grid)
        y = rayon * np.sin(theta_grid)
        self.ax.plot_surface(x, y, z_grid, color=col, alpha=self.opac, edgecolor='none')
    
        # Bases (inférieure et supérieure)
        r = np.linspace(0, rayon, 20)
        theta_base = np.linspace(0, 2*np.pi, n_theta)
        theta_b, r_b = np.meshgrid(theta_base, r)
        x_base = r_b * np.cos(theta_b)
        y_base = r_b * np.sin(theta_b)
    
        # Base inférieure
        self.ax.plot_surface(x_base, y_base, np.zeros_like(x_base), color=col, alpha=self.opac)
        # Base supérieure
        self.ax.plot_surface(x_base, y_base, np.full_like(x_base, hauteur), color=col, alpha=self.opac)
    
        """# Petit repère
        repere_pts = np.array([
            [0,0,0],
            [rayon*0.2,0,0],
            [0,rayon*0.2,0],
            [0,0,hauteur*0.2]
        ])
        lignes = [[repere_pts[0], repere_pts[1]],
                  [repere_pts[0], repere_pts[2]],
                  [repere_pts[0], repere_pts[3]]]
        self.ax.add_collection3d(Poly3DCollection(lignes, edgecolor='blue', linewidth=1.2, alpha=1))"""
    
        # Limites et aspect
        self.ax.set_xlim(-rayon, rayon)
        self.ax.set_ylim(-rayon, rayon)
        self.ax.set_zlim(0, hauteur)
        self.ax.set_box_aspect([1, 1, hauteur/rayon])

        # Label 
        self.ax.set_xlabel('X (mm)')
        self.ax.set_ylabel('X (mm)')
        self.ax.set_zlabel('X (mm)')
        
    
        # Affichage
        if hasattr(self, 'canvas'):
            self.canvas.draw()
            self.afficher_reperes(taille=self.hauteur)
        else:
            plt.show()


    def Clear(self):
        # Reinitialisation des États logique de la simulation
        # --- Géométrie
        self.geo_choosing = None
        self.longueur = None
        self.largeur = None
        self.hauteur = None
        self.rayon = None
    
        # --- Matériau
        self.materiau = None
        self.module_dYoung = None
        self.coeff_poisson = None
        self.masse_volumique = None
    
        # --- Sollicitation
        self.sollicitation = None
        self.charge = None
    
        # --- Maillage
        self.noeuds = None
        self.elements = None
        self.opac = 0.2
        
        # Suppression du graphique précédent
        if hasattr(self, 'ax'):
            self.ax.cla()
            self.ax.axis('off')
    
        # Supprimer dimensions si elles existent
        for attr in ['longueur', 'largeur', 'hauteur', 'rayon']:
            if hasattr(self, attr):
                delattr(self, attr)
    
        # Supprimer maillage
        for attr in ['noeuds', 'elements']:
            if hasattr(self, attr):
                delattr(self, attr)
    def Mesh(self):
        #--- Vérifie qu'un solide a été créé
        if self.geo_choosing == "Cylindre" and not hasattr(self, 'rayon'):
            messagebox.showwarning("Attention", "Créez d'abord un solide!")
            return
        if self.geo_choosing == "Pavé" and not hasattr(self, 'longueur'):
            messagebox.showwarning("Attention", "Créez d'abord un solide!")
            return
        
        #---Efface le graphique précédent
        self.ax.clear()
        
        if self.geo_choosing == "Cylindre":
            self.mailler_cylindre()
        else:
            self.mailler_pave()
        
        #---Configurer la vue
        self.ax.axis('off')
        self.canvas.draw()
        
        messagebox.showinfo("Maillage", f"Maillage terminé!\nNombre de nœuds: {len(self.noeuds)}")
   
    # ===========================================
    # GENERATION DU MAILLAGE
    # ===========================================
    
    def mailler_pave(self, n_x=12, n_y=12, n_z=12):
        """
        Génère un maillage quadrilatéral pour un pavé et l'affiche.
        Paramètres :
        n_x, n_y, n_z : nombre de subdivisions dans chaque direction.
        """
        if self.longueur <= 0 or self.largeur <= 0 or self.hauteur <= 0:
            messagebox.showerror("Problème", "Dimensions invalides")
            return
    
        # --- Création des nœuds ---
        self.noeuds = []
        node_map = {}
        node_index = 0
        for k in range(n_z + 1):
            z = k * self.hauteur / n_z
            for j in range(n_y + 1):
                y = j * self.largeur / n_y
                for i in range(n_x + 1):
                    x = i * self.longueur / n_x
                    self.noeuds.append([x, y, z])
                    node_map[(i, j, k)] = node_index
                    node_index += 1
        self.noeuds = np.array(self.noeuds)
    
        # --- Définition des quadrilatères ---
        self.elements = []
    
        # Face avant (y = 0)
        for k in range(n_z):
            for i in range(n_x):
                n1 = node_map[(i, 0, k)]
                n2 = node_map[(i+1, 0, k)]
                n3 = node_map[(i+1, 0, k+1)]
                n4 = node_map[(i, 0, k+1)]
                self.elements.append([n1, n2, n3, n4])
    
        # Face arrière (y = largeur)
        for k in range(n_z):
            for i in range(n_x):
                n1 = node_map[(i, n_y, k)]
                n2 = node_map[(i+1, n_y, k)]
                n3 = node_map[(i+1, n_y, k+1)]
                n4 = node_map[(i, n_y, k+1)]
                self.elements.append([n1, n2, n3, n4])
    
        # Face gauche (x = 0)
        for k in range(n_z):
            for j in range(n_y):
                n1 = node_map[(0, j, k)]
                n2 = node_map[(0, j+1, k)]
                n3 = node_map[(0, j+1, k+1)]
                n4 = node_map[(0, j, k+1)]
                self.elements.append([n1, n2, n3, n4])
    
        # Face droite (x = longueur)
        for k in range(n_z):
            for j in range(n_y):
                n1 = node_map[(n_x, j, k)]
                n2 = node_map[(n_x, j+1, k)]
                n3 = node_map[(n_x, j+1, k+1)]
                n4 = node_map[(n_x, j, k+1)]
                self.elements.append([n1, n2, n3, n4])
    
        # Face bas (z = 0)
        for j in range(n_y):
            for i in range(n_x):
                n1 = node_map[(i, j, 0)]
                n2 = node_map[(i+1, j, 0)]
                n3 = node_map[(i+1, j+1, 0)]
                n4 = node_map[(i, j+1, 0)]
                self.elements.append([n1, n2, n3, n4])
    
        # Face haut (z = hauteur)
        for j in range(n_y):
            for i in range(n_x):
                n1 = node_map[(i, j, n_z)]
                n2 = node_map[(i+1, j, n_z)]
                n3 = node_map[(i+1, j+1, n_z)]
                n4 = node_map[(i, j+1, n_z)]
                self.elements.append([n1, n2, n3, n4])
    
        # --- Affichage des quadrilatères ---
        self.ax.cla()
        self.ax.set_axis_off()
        for quad in self.elements:
            coords = self.noeuds[quad]
            collection = Poly3DCollection([coords], facecolor='blue', edgecolor='black', alpha=1, linewidth=0.7)
            self.ax.add_collection3d(collection)
    
        # --- Ajouter repère visuel ---
        """points = np.array([[-2, 0, 0], [-1, 0, 0], [-2, 1, 0], [-2, 0, 1]])
        ligne = [[points[0], points[1]], [points[0], points[2]], [points[0], points[3]]]
        repere = Poly3DCollection(ligne, edgecolor='blue', linewidth=1.2, alpha=1)
        self.ax.add_collection3d(repere)"""
    
        # --- Limites des axes ---
        self.ax.set_xlim([0, self.longueur])
        self.ax.set_ylim([0, self.largeur])
        self.ax.set_zlim([0, self.hauteur])
    
        # Proportions correctes
        max_dim = max(self.longueur, self.largeur, self.hauteur)
        self.ax.set_box_aspect([self.longueur/max_dim, self.largeur/max_dim, self.hauteur/max_dim])
        #self.afficher_reperes(taille=self.longueur)
        plt.show()


    def mailler_cylindre(self, n_theta=20, n_z=10):
       
        """
        Génère un maillage surfacique d’un cylindre.
        - Nœuds circulaires
        - Éléments quadrilatères
        """
        
        self.noeuds = []
        self.elements = []
    
        # Création des nœuds
        for k in range(n_z + 1):
            z = k * self.hauteur / n_z
            for i in range(n_theta):
                theta = 2 * np.pi * i / n_theta
                x = self.rayon * np.cos(theta)
                y = self.rayon * np.sin(theta)
                self.noeuds.append([x, y, z])
    
        self.noeuds = np.array(self.noeuds)
    
        # Création des quadrilatères pour la surface latérale
        for k in range(n_z):
            for i in range(n_theta):
                n1 = k * n_theta + i
                n2 = k * n_theta + (i + 1) % n_theta
                n3 = (k + 1) * n_theta + (i + 1) % n_theta
                n4 = (k + 1) * n_theta + i
                self.elements.append([n1, n2, n3, n4])
    
        # Création des faces supérieures et inférieures
        center_bottom = len(self.noeuds)
        self.noeuds = np.vstack([self.noeuds, [0, 0, 0]])  # centre bas
        center_top = len(self.noeuds)
        self.noeuds = np.vstack([self.noeuds, [0, 0, self.hauteur]])  # centre haut
    
        # Face inférieure
        for i in range(n_theta):
            n1 = i
            n2 = (i + 1) % n_theta
            self.elements.append([center_bottom, n1, n2])
    
        # Face supérieure
        for i in range(n_theta):
            n1 = n_theta * n_z + i
            n2 = n_theta * n_z + (i + 1) % n_theta
            self.elements.append([center_top, n2, n1])
    
        # --- Affichage ---
        self.ax.cla()
        self.ax.set_axis_off()
    
        for quad in self.elements:
            face = [self.noeuds[idx] for idx in quad]
            self.ax.add_collection3d(Poly3DCollection([face], facecolor='blue', edgecolor='black', alpha=self.opac))
    
        # Limites et aspect
        self.ax.set_xlim(-self.rayon, self.rayon)
        self.ax.set_ylim(-self.rayon, self.rayon)
        self.ax.set_zlim(0, self.hauteur)
        self.ax.set_box_aspect([1, 1, self.hauteur/self.rayon])
        #self.afficher_reperes(taille=self.hauteur)

        plt.show()

    # ===========================================
    # APPLICATION DES DEFORMATIONS
    # ===========================================
    def appliquer_deformation(self):

        """
        Applique une déformation axiale simulée :
        - Calcule contrainte σ = F/A
        - Déformation ε = σ/E
        - Déplacement des nœuds
        - Affichage couleur gradient
        """
        
        if len(self.noeuds) == 0:
            messagebox.showwarning("Attention", "Créez d'abord un maillage!")
            return
        else:
            td = time()
            F = self.charge
            d = int(self.conditions_limites["face"])
            E = self.module_dYoung * 1e3
            nu = self.coeff_poisson
            if self.sollicitation == "Compression":
                F = -F
            noeuds = self.noeuds.copy()
            noeuds_def, deformation = self._deformer(F, noeuds, d, E, nu, self.aire)
            deformation = np.sqrt(deformation[:, 0]**2 + deformation[:, 1]**2 + deformation[:, 2]**2)
            
            self.ax.clear()
            self.noeuds = noeuds_def


            vmin, vmax = 0.0, max(deformation)
            norm = colors.Normalize(vmin = 0, vmax = vmax)
            cmap = plt.cm.RdYlGn_r


            for quad in self.elements:
                face = [self.noeuds[idx] for idx in quad]
                list_def_quad =[deformation[idx] for idx in quad]
                def_quad = sum(list_def_quad)/len(list_def_quad)
                color = cmap(norm(def_quad))
                self.ax.add_collection3d(Poly3DCollection([face], facecolor = color, edgecolor='black', alpha=self.opac))
            tf = time()
            temp = round(tf-td, 2)
            """
            for quad in self.elements:
                face = [self.noeuds[idx] for idx in quad]
                self.ax.add_collection3d(Poly3DCollection([face], facecolor='green', edgecolor='black', alpha=self.opac))
            """
                
            messagebox.showinfo("Done", f" Déformation appliquée ✅\n Temps total de simulation {temp} secondes")
            
            self.ax.set_xlabel('X (mm)')
            self.ax.set_ylabel('Y (mm)')
            self.ax.set_zlabel('Z (mm)')
            self.ax.set_axis_off()
            self.canvas.draw()
            
            #except:
                #messagebox.showerror("Erreur", "Paramètres invalides!")
    
    def _deformer(self, F, noeuds, direction, E, nu, A, clamp_length_ratio=0.35):

        """
        Calcule le champ de déplacement nodal :
        - Déformation longitudinale
        - Contraction transverse (effet de Poisson)
        - Retourne les nouveaux nœuds
        """
        
        dim = noeuds.shape[1]
        u = np.zeros_like(noeuds, dtype=float)
     
        sigma = F / A
        eps = sigma / E
    
        x = noeuds[:, direction]
        x_min = x.min()
        x_max = x.max()
        L = x_max - x_min
     
        lc = clamp_length_ratio * L
    
        s = (x - x_min) / lc
        s = np.clip(s, 0.0, 1.0)
        
        w= 1 - np.exp(-4*s)
    
        for i in range(dim):
            if i == direction:
                u[:, i] = eps * (x - x_min)
            else:
                u[:, i] = -nu * eps * (noeuds[:, i] - noeuds[:, i].mean())
    
        u *= w[:, None]
    
        return noeuds + u, u

    # ===========================================
    # LANCAMENT DE L'APPLICATION
    # ===========================================
    def Go(self):

        """
        Initialise le style graphique
        et lance la boucle principale Tkinter.
        """
        
        style = ttk.Style()
        style.theme_use("alt")
        style.configure("Primary.TButton",
                        font="Segoe UI 11 bold",
                        fg="white",
                        bg="black",
                        padding=10,
                        relief="solid",
                        borderwidth=5,
                        width=15)
        style.map("Primary.TButton", bg=[("active", "dim gray")], fg=[("active", "white")])
    
        ttk.Button(self.frame_principal, text="LANCER L'APPLICATION", 
                   command=self.Mode, width=30, cursor="hand2").pack(pady=30)
        ttk.Button(self.frame_principal, text="FERMER", 
                   command=self.interface.destroy, width=20, cursor="hand2").pack(side="bottom", pady=30)

        self.interface.mainloop() 

#### POINT D'ENTREE DU PROGRAMME

In [4]:
sl = simulation()
sl.Go()

###### Wrote by GENIES-O